In [18]:
import os
import numexpr
from datetime import datetime
from docx import Document

In [19]:
# Modern LangChain imports
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

In [20]:
# ==========================================
# 1. Define the 3 Core Tools
# ==========================================

# Tool 1: Arithmetic Solver
@tool
def calculate(expression: str) -> str:
    """
    Evaluates mathematical expressions safely. 
    Input must be a valid mathematical expression (e.g., '45 * 32' or '100 / 4').
    """
    try:
        result = numexpr.evaluate(expression)
        return str(result.item())
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Tool 2: Wikipedia Extractor
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000),
    description="Searches Wikipedia for factual information and summaries. Input should be a specific topic."
)

# Tool 3: PDF Document Creator
@tool
def create_word_doc(content: str) -> str:
    """
    Generates a Word document (.docx) from the provided text string. 
    Input should be the final, nicely formatted text you want written into the document.
    """
    folder = "generated_docs"
    os.makedirs(folder, exist_ok=True)
    
    filename = f"Summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
    filepath = os.path.join(folder, filename)
    
    # Create and save the Word document
    doc = Document()
    doc.add_paragraph(content)
    doc.save(filepath)
    
    return f"Word document successfully created and saved at: {filepath}"

# Bundle the tools together
tools = [calculate, wikipedia_tool, create_word_doc]

In [21]:
# ==========================================
# 2. Initialize LLM and Agent Framework
# ==========================================

# Connect to your local LM Studio instance
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio", # Bypasses the OpenAI Key requirement
    temperature=0.2
)

# Define the precise routing logic based on your 3 requirements
prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are an Agentic AI that processes user input and takes action based on the following rules:\n"
        "1. If the input is an arithmetic problem, use the 'calculate' tool to solve it.\n"
        "2. If the user asks for information on a topic, use the 'wikipedia' tool to extract it.\n"
        "3. If the user asks to create a Word document (or says 'make it a file'), you MUST use the 'create_word_doc' tool. "
        "If they ask to base the file on 'earlier' or 'previous' topics, find that exact information in the CHAT HISTORY below and pass it to the tool.\n\n"
        "IMPORTANT: Do not ever say you lack access to previous conversations. They are explicitly provided to you below.\n\n"
        "--- CHAT HISTORY ---\n"
        "{chat_history}\n"
        "--------------------"
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [22]:
# ==========================================
# 3. Interactive User Chat Loop with Memory
# ==========================================

if __name__ == "__main__":
    print("=====================================================")
    print("Agentic AI Initialized. Type 'exit' or 'quit' to stop.")
    print("=====================================================\n")
    
    # Initialize an empty list to store the conversation history
    chat_history = []
    
    while True:
        user_input = input("\nYou: ")
        
        if user_input.lower() in ['exit', 'quit']:
            print("Shutting down Agentic AI. Goodbye!")
            break
            
        if not user_input.strip():
            continue
            
        print("\nAgent is thinking...")
        
        try:
            # Pass both the input AND the chat history into the agent
            response = agent_executor.invoke({
                "input": user_input,
                "chat_history": chat_history
            })
            
            # --- MEMORY UPDATE ---
            # Save the back-and-forth so the AI remembers it for the next loop
            chat_history.append(HumanMessage(content=user_input))
            chat_history.append(AIMessage(content=response["output"]))
            # ---------------------
            
            print("\n" + "="*50)
            print(f"USER INPUT  : {user_input}")
            print("-" * 50)
            print(f"FINAL OUTPUT:\n{response['output']}")
            print("="*50)
            
        except Exception as e:
            print(f"\nAn error occurred: {e}")

Agentic AI Initialized. Type 'exit' or 'quit' to stop.


Agent is thinking...


> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'token in artificial intelligence'}`
responded: 



Page: Large language model
Summary: A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can generate, summarize, translate and parse text in many contexts, and are a foundational technology behind modern chatbots. Biased or inaccurate training data can make an LLM's output less reliable. 
As of 2024, the largest and most capable LLMs are all based on transformer architectures, which, according to the 2017 paper Attention Is All You Need, can be more efficient and parallelizable than earlier statistical and recurrent neural network models. Research into other architectures, such as state space models, is ongoing.
Benchmark evaluations for LLMs attempt to measure model reasoning